In [ ]:
# ================= Cell 1: 导入库和初始化 =================
import vertexai
from vertexai.generative_models import GenerativeModel
import PyPDF2
import json
import os
import pandas as pd
import google.auth
import re # 导入正则表达式库，用于清理文本

# --- 自动初始化配置 ---
# 在Vertex AI Workbench中，这会自动获取你的项目ID和认证信息
try:
    credentials, project_id = google.auth.default()
    location = 'us-central1' # 推荐使用这个区域
    vertexai.init(project=project_id, location=location)
    print(f"✅ GCP Project ID: {project_id}")
    print(f"✅ Location: {location}")
    print("✅ Vertex AI 初始化成功！一切就绪，可以开始工作了。")
except Exception as e:
    print(f"❌ 初始化失败: {e}")
    print("请确保您是在Vertex AI Workbench环境中运行，并且Vertex AI API已经启用。")

In [ ]:
# ================= Cell 2: 定义所有功能函数 =================

def extract_text_from_pdf(pdf_path):
    """从PDF文件中安全地提取文本内容。"""
    print(f"⏳ 正在读取PDF文件: {pdf_path}...")
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for i, page in enumerate(pdf_reader.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text
                else:
                    print(f"  - 警告: 第 {i+1} 页未能提取到文本。")
        # 清理文本中的多余换行符和奇怪的空格
        text = re.sub(r'\s+', ' ', text).strip()
        print(f"✅ PDF文本提取完毕，总计约 {len(text)} 个字符。")
        return text
    except Exception as e:
        print(f"❌ 读取PDF时出错: {e}")
        return None

def extract_data_with_gemini(text_content, your_prompt):
    """调用Gemini API，并使用你提供的Prompt进行数据提取。"""
    print("⏳ 正在连接Gemini，发送请求... 这可能需要1-2分钟，请耐心等待。")
    try:
        # 选择模型
        model = GenerativeModel("gemini-1.0-pro-vision") # 使用vision模型，它对解析PDF布局的文本更友好

        # 组合Prompt和文本
        full_prompt = your_prompt.format(text_content=text_content)

        # 调用API
        response = model.generate_content(
            full_prompt,
            generation_config={
                "temperature": 0.1, # 较低的温度确保结果更稳定、更具确定性
                "max_output_tokens": 8192 # 设置较大的输出上限
            }
        )
        
        print("✅ 已收到Gemini的回复。")
        # 清理回复，只保留JSON部分
        cleaned_response = response.text.strip().lstrip("```json").rstrip("```")
        return cleaned_response
    except Exception as e:
        print(f"❌ 调用Gemini API时出错: {e}")
        return None

def process_and_save_to_csv(json_string, output_csv_path):
    """解析JSON字符串，将其转换为扁平化的表格，并保存为CSV文件。"""
    print("⏳ 正在解析数据并生成表格...")
    try:
        # 将JSON字符串转换为Python字典
        data_dict = json.loads(json_string)
        
        # 提取全局信息
        study_info = data_dict.get('study_info', {})
        moderators = data_dict.get('moderator_variables', {})
        
        flat_moderators = {
            'first_author': study_info.get('first_author'), 'year': study_info.get('year'),
            'site_name': moderators.get('location', {}).get('site_name'),
            'coordinates': moderators.get('location', {}).get('coordinates'),
            'country': moderators.get('location', {}).get('country'),
            'mean_annual_temp_c': moderators.get('climate', {}).get('mean_annual_temperature_c'),
            'mean_annual_precip_mm': moderators.get('climate', {}).get('mean_annual_precipitation_mm'),
            'soil_type_wrb': moderators.get('soil_background', {}).get('soil_type_wrb'),
            'soil_texture': moderators.get('soil_background', {}).get('texture'),
            'initial_ph': moderators.get('soil_background', {}).get('initial_ph'),
            'initial_soc_g_kg': moderators.get('soil_background', {}).get('initial_soc_g_kg'),
            'exp_duration_years': moderators.get('experimental_design', {}).get('duration_years'),
            'treatments_overview': "; ".join([f"{t.get('name', '')}: {t.get('description', '')}" for t in moderators.get('experimental_design', {}).get('treatments_overview', [])])
        }

        all_rows_data = []
        response_variables = data_dict.get('response_variables', [])
        
        if not response_variables:
            print("⚠️ 警告: Gemini的回复中没有找到 'response_variables' 数据点。CSV文件将只包含主持人变量。")
            # 即使没有因变量，也可能希望保存主持人变量信息
            all_rows_data.append(flat_moderators)
        else:
            for record in response_variables:
                row_data = flat_moderators.copy()
                row_data.update({
                    'variable_name': record.get('variable_name'), 'unit': record.get('unit'),
                    'treatment_group': record.get('treatment_group'), 'control_group': record.get('control_group'),
                    'mean_t': record.get('mean_t'), 'sd_t': record.get('sd_t'), 'n_t': record.get('n_t'),
                    'mean_c': record.get('mean_c'), 'sd_c': record.get('sd_c'), 'n_c': record.get('n_c'),
                    'source': record.get('context', {}).get('source'),
                    'sampling_depth_cm': record.get('context', {}).get('sampling_depth_cm'),
                    'measurement_year': record.get('context', {}).get('measurement_year')
                })
                all_rows_data.append(row_data)

        # 使用Pandas创建DataFrame并保存
        df = pd.DataFrame(all_rows_data)
        df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
        print(f"✅ 数据已成功保存到CSV文件: {output_csv_path}")
        return df # 返回DataFrame方便在Notebook中直接预览

    except json.JSONDecodeError:
        print("❌ 解析JSON失败！请检查Gemini返回的原始文本，它可能不是一个有效的JSON格式。")
        print("--- Gemini原始回复 ---")
        print(json_string)
        return None
    except Exception as e:
        print(f"❌ 处理或保存文件时出错: {e}")
        return None

In [ ]:
# ================= Cell 3: 定义你的提取指令 (Prompt) =================

# 这是你告诉AI要做什么的核心指令。你可以根据你的变量表头来修改这个模板。
# 【重要】请仔细检查JSON结构中的字段名，确保它们和你想要的一致。

PROMPT_TEMPLATE = """
你是一位顶尖的土壤生态学研究员，正在为一项Meta分析进行精确的数据提取。你的任务是阅读下面的论文全文，并严格按照我提供的JSON格式，提取所有相关信息。

**规则:**
1.  **精确匹配:** 严格按照下方JSON的键名(key)和结构进行输出。
2.  **完整性:** 尽可能填写所有字段。如果信息确实未在论文中提及，请使用 `null` 或 "Not Mentioned"。
3.  **数据来源:** 对于每个因变量数据点，务必在`context`中指明其来源（如 "Table 2" 或 "Figure 3"）。
4.  **配对数据:** 因变量数据必须是成对的（处理组 vs 对照组）。如果某个处理找不到对应的对照组数据，请不要记录该条目。
5.  **输出格式:** 最终输出必须是一个干净的、可以被直接解析的JSON对象，不要在JSON代码块前后添加任何额外的解释性文字。

**JSON输出结构要求:**
{{
  "study_info": {{
    "first_author": "论文第一作者姓氏",
    "year": "发表年份"
  }},
  "moderator_variables": {{
    "location": {{
      "site_name": "研究地点的具体名称",
      "coordinates": "经纬度 (格式: '纬度, 经度')",
      "country": "国家"
    }},
    "climate": {{
      "mean_annual_temperature_c": "年均温（仅数字）",
      "mean_annual_precipitation_mm": "年均降水量（仅数字）"
    }},
    "soil_background": {{
      "soil_type_wrb": "土壤类型（按WRB或USDA ST分类）",
      "texture": "土壤质地 (如: Loam, Sandy Loam)",
      "initial_ph": "初始土壤pH值",
      "initial_soc_g_kg": "初始土壤有机碳含量 (单位: g/kg)"
    }},
    "experimental_design": {{
      "duration_years": "实验持续年限（仅数字）",
      "treatments_overview": [
        {{ "name": "处理1的缩写 (如: CK)", "description": "处理1的简短描述" }},
        {{ "name": "处理2的缩写 (如: NPK)", "description": "处理2的简短描述" }}
      ]
    }}
  }},
  "response_variables": [
    {{
      "variable_name": "因变量的名称 (如: soil_organic_carbon)",
      "unit": "该变量的单位 (如: g/kg)",
      "treatment_group": "处理组的缩写 (必须在上面overview中出现)",
      "control_group": "对照组的缩写 (必须在上面overview中出现)",
      "mean_t": "处理组的均值 (仅数字)",
      "sd_t": "处理组的标准差 (仅数字)",
      "n_t": "处理组的样本量 (仅数字)",
      "mean_c": "对照组的均值 (仅数字)",
      "sd_c": "对照组的标准差 (仅数字)",
      "n_c": "对照组的样本量 (仅数字)",
      "context": {{
        "source": "数据来源 (如: Table 2)",
        "sampling_depth_cm": "采样深度 (如: 0-15)",
        "measurement_year": "测量年份 (如果提及)"
      }}
    }}
  ]
}}

--- 以下是论文文本 ---
{text_content}
"""

print("✅ Prompt模板已定义。请检查上面的JSON结构是否符合您的要求。")

In [ ]:
# ================= Cell 4: 主执行程序 =================

# --- 1. 在这里设置你的文件名 ---
pdf_file_name = "my_paper.pdf"  # 🔴🔴🔴 <--- 请务必修改成你上传的PDF文件名!

# --- 2. 准备工作 ---
output_csv_name = f"extracted_{os.path.splitext(pdf_file_name)[0]}.csv"

if not os.path.exists(pdf_file_name):
    print(f"❌ 错误：找不到文件 '{pdf_file_name}'。请确保文件名正确，且文件已上传到左侧文件列表中。")
else:
    # --- 3. 执行流程 ---
    # 第一步: 读取PDF
    paper_text = extract_text_from_pdf(pdf_file_name)
    
    if paper_text:
        # 为了防止超出token限制，做一个安全截断
        # 32k token ~ 12万字符。我们留一些余地给prompt。
        max_chars = 100000 
        if len(paper_text) > max_chars:
            print(f"⚠️ 警告: 论文过长，已截取前 {max_chars} 个字符进行分析。")
            paper_text = paper_text[:max_chars]
            
        # 第二步: 调用Gemini
        gemini_output_json = extract_data_with_gemini(paper_text, PROMPT_TEMPLATE)
        
        if gemini_output_json:
            # 第三步: 处理数据并保存
            final_dataframe = process_and_save_to_csv(gemini_output_json, output_csv_name)
            
            # --- 4. 在Notebook中直接预览结果 ---
            if final_dataframe is not None:
                print("\n🎉🎉🎉 提取成功！以下是生成表格的预览：")
                display(final_dataframe) # display() 是在Jupyter中漂亮地展示表格的命令